# 4.1 同一個字放不同位置，怎樣提供不同線索？


「貓追狗」與「狗追貓」字相同、順序不同。字特徵只靠 ID 查表，單看「貓」那一排不知道它放在哪格。因此除了文字內容，我們再提供位置線索。

先用「貓看貓」，約定 ID 1=貓、2=看。兩次貓都查到 `[1,0,0]`；再讓位置 0、1、2 各查一排三數，與字向量逐格相加。這叫**位置表示**，本節使用可調的絕對位置表。

| 位置 | 字特徵 | 本節指定的位置特徵 | 相加後 |
| --- | --- | --- | --- |
| 0／貓 | `[1,0,0]` | `[0,0,0]` | `[1,0,0]` |
| 1／看 | `[0,1,0]` | `[0.1,0.2,0.3]` | `[0.1,1.2,0.3]` |
| 2／貓 | `[1,0,0]` | `[0.2,0.4,0.6]` | `[1.2,0.4,0.6]` |


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

e = nn.Embedding(4, 3)
p = nn.Embedding(5, 3)
with torch.no_grad():
    e.weight.copy_(torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]))
    p.weight.copy_(torch.tensor([[0.0, 0.0, 0.0], [0.1, 0.2, 0.3], [0.2, 0.4, 0.6], [0.3, 0.6, 0.9], [0.4, 0.8, 1.2]]))
ids = torch.tensor([1, 2, 1])
positions = torch.arange(3)
x = e(ids) + p(positions)
print(positions)
print(x)
assert not torch.equal(x[0], x[2])

`e` 是字表，`p` 是位置表。`torch.arange(3)` 產生 `[0,1,2]`，選位置表的相應列；`e(ids)+p(positions)` 正是表中的逐格相加。最後檢查兩次貓的相加結果不同。

這些位置小數由人指定，沒有「位置越後，每格必須等差增加」的要求。真正使用這種模型時，位置表通常從亂數開始，與字表一起訓練。加入位置使線索可用，後面的計算還要學會使用它，不代表相加後已懂語序。

因果許可決定誰能讀誰，位置特徵給每位置額外數字，兩者用途不同。即使模型能從許可範圍取得一些順序線索，本節也讓位置明確參與特徵計算。

練習在填表的 `with torch.no_grad():` 區塊加 `p.weight.zero_()`，並把斷言改成 `assert torch.equal(x[0],x[2])`。兩次貓又相同，差異正是位置表帶來的。

位置表有五列，只容許位置 0 到 4。設定能放五格，只表示這些索引有效；某位置是否被充分訓練、長文是否能用好線索，仍是別的問題。

<details>
<summary>補充與重做</summary>

字特徵見 [2.1](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html)，逐格相加要兩表同寬。本例輸入位置 5 會超出表格範圍。`no_grad()` 的縮排範圍只包手動修改，見 [1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)；輸出附的 `AddBackward0` 是相加的求導記錄。後面的 RoPE 與視窗延伸會改變位置方法，本節先建立查表起點。

</details>
